# Parte 2 — Diagnóstico, limpeza e validação de domínio

| | |
|---|---|
| **Entrada** | os três CSVs em `dados_brutos/` |
| **Saída desta parte** | `df_limpo` (DataFrame em memória) + relatório de qualidade |

> Notebook executado com *Restart & Run All* sem erro. Caminhos relativos; nenhum CSV original é sobrescrito.

**Escopo:** 2.1 diagnóstico (nulos, duplicatas), 2.2 conversão de tipos, 2.3 padronização de
textos com validação contra o dicionário da PRF, 2.4 remoção das colunas de localidade fina.


## 0. Configuração do ambiente

Resolve a raiz do projeto por `Path` (funciona aberto da raiz do repositório ou de
`notebooks/`) e define as constantes usadas nas partes.


In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)


def localizar_raiz() -> Path:
    """Sobe a partir do diretório atual até achar a pasta de dados brutos."""
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "dados_brutos").is_dir():
            return candidato
    raise FileNotFoundError(
        "Não encontrei a pasta 'dados_brutos/'. "
        "Abra o notebook a partir da raiz do projeto ou de 'notebooks/'."
    )


RAIZ = localizar_raiz()
DIR_BRUTOS = RAIZ / "dados_brutos"
DIR_TRATADOS = RAIZ / "dados_tratados"
DIR_RESULTADOS = RAIZ / "resultados"

for pasta in (DIR_TRATADOS, DIR_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

SEPARADOR = ";"
ENCODING_SAIDA = "utf-8-sig"
ENCODINGS_TENTATIVA = ("utf-8", "latin1", "utf-8-sig", "cp1252")
ANOS = (2023, 2024, 2025)

print(f"Raiz do projeto : {RAIZ}")
print(f"Dados brutos    : {DIR_BRUTOS}")
print(f"Dados tratados  : {DIR_TRATADOS}")
print(f"Resultados      : {DIR_RESULTADOS}")


Raiz do projeto : /home/user/PRF-pandas-e-Plotly---2023-2025
Dados brutos    : /home/user/PRF-pandas-e-Plotly---2023-2025/dados_brutos
Dados tratados  : /home/user/PRF-pandas-e-Plotly---2023-2025/dados_tratados
Resultados      : /home/user/PRF-pandas-e-Plotly---2023-2025/resultados


## 2.0 Reconstrução da base consolidada

Este notebook é autônomo: refaz a leitura e a concatenação da Parte 1 para poder rodar isolado.


In [2]:
def ler_csv_prf(caminho: Path, sep: str = SEPARADOR) -> pd.DataFrame:
    """Lê um CSV da PRF tentando vários encodings em sequência.

    Diferença em relação à versão do Módulo 04: um encoding que falha não
    interrompe a função (não há 'raise' dentro do laço); a próxima tentativa
    é executada e o erro só é levantado se todas falharem.
    """
    ultimo_erro: Exception | None = None
    for encoding in ENCODINGS_TENTATIVA:
        try:
            df = pd.read_csv(
                caminho, sep=sep, encoding=encoding, low_memory=False,
                dtype={"id": "string"},
            )
        except (UnicodeDecodeError, UnicodeError) as erro:
            ultimo_erro = erro
            continue
        print(f"{caminho.name}: lido com encoding={encoding!r} "
              f"({len(df):,} linhas x {df.shape[1]} colunas)")
        return df
    raise RuntimeError(
        f"Nenhum encoding de {ENCODINGS_TENTATIVA} funcionou para {caminho}"
    ) from ultimo_erro

def normalizar_nome_coluna(nome: str) -> str:
    """minúsculas, sem acento, separador único em '_'."""
    nome = str(nome).strip().lower()
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("utf-8")
    for antigo, novo in ((" ", "_"), ("-", "_"), ("/", "_")):
        nome = nome.replace(antigo, novo)
    while "__" in nome:
        nome = nome.replace("__", "_")
    return nome.strip("_")

dados_por_ano = {}
for ano in ANOS:
    df = ler_csv_prf(DIR_BRUTOS / f"datatran{ano}.csv")
    df.columns = [normalizar_nome_coluna(c) for c in df.columns]
    df["arquivo_origem"] = ano
    dados_por_ano[ano] = df

acidentes_prf_historico = pd.concat(
    [dados_por_ano[ano] for ano in ANOS], ignore_index=True, sort=False
)
linhas_na_integracao = len(acidentes_prf_historico)

assert linhas_na_integracao == sum(len(d) for d in dados_por_ano.values())
print(f"\nBase consolidada: {acidentes_prf_historico.shape}")


datatran2023.csv: lido com encoding='latin1' (67,766 linhas x 30 colunas)


datatran2024.csv: lido com encoding='latin1' (73,202 linhas x 30 colunas)


datatran2025.csv: lido com encoding='latin1' (72,529 linhas x 30 colunas)

Base consolidada: (213497, 31)


## 2.1 Diagnóstico: `info()`, nulos e duplicidades

**Pergunta:** o que a base tem de sujeira estrutural?

O diagnóstico vem **antes** de qualquer alteração, para que o relatório descreva o dado
original e não o já tratado.


In [3]:
print("=" * 70)
print("INFO")
print("=" * 70)
acidentes_prf_historico.info(memory_usage="deep", show_counts=True)

print("\n" + "=" * 70)
print("TIPOS DE DADOS")
print("=" * 70)
resumo_tipos = (
    acidentes_prf_historico.dtypes.astype(str)
    .value_counts().rename_axis("tipo").reset_index(name="qtd_colunas")
)
display(resumo_tipos)


INFO


<class 'pandas.DataFrame'>
RangeIndex: 213497 entries, 0 to 213496
Data columns (total 31 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   id                      213497 non-null  string
 1   data_inversa            213497 non-null  str   
 2   dia_semana              213497 non-null  str   
 3   horario                 213497 non-null  str   
 4   uf                      213497 non-null  str   
 5   br                      213497 non-null  int64 
 6   km                      213497 non-null  str   
 7   municipio               213497 non-null  str   
 8   causa_acidente          213497 non-null  str   
 9   tipo_acidente           213497 non-null  str   
 10  classificacao_acidente  213494 non-null  str   
 11  fase_dia                213497 non-null  str   
 12  sentido_via             213497 non-null  str   
 13  condicao_metereologica  213497 non-null  str   
 14  tipo_pista              213497 non-null  str   

,tipo,qtd_colunas
0,str,20
1,int64,10
2,string,1


In [4]:
nulos = pd.DataFrame({
    "qtd_nulos": acidentes_prf_historico.isna().sum(),
    "perc_nulos": acidentes_prf_historico.isna().mean().mul(100).round(4),
}).sort_values("perc_nulos", ascending=False)

print(f"Colunas com pelo menos um nulo: {(nulos['qtd_nulos'] > 0).sum()} de {len(nulos)}")
display(nulos[nulos["qtd_nulos"] > 0])


Colunas com pelo menos um nulo: 4 de 31


,qtd_nulos,perc_nulos
uop,155,0.0726
delegacia,91,0.0426
regional,19,0.0089
classificacao_acidente,3,0.0014


### Duplicidades exatas

**Pergunta:** existem linhas 100% idênticas?

A remoção é executada mesmo que o resultado seja zero — é o item pedido pelo exercício, e o
notebook declara explicitamente quantas linhas saíram. Nesta base os três arquivos já vêm sem
duplicatas exatas, então a operação remove **0 linhas** e o total não muda.


In [5]:
qtd_duplicadas = int(acidentes_prf_historico.duplicated().sum())
print(f"Duplicidades exatas encontradas: {qtd_duplicadas:,}")

df_deduplicado = acidentes_prf_historico.drop_duplicates().copy()
removidas = len(acidentes_prf_historico) - len(df_deduplicado)

print(f"Linhas removidas              : {removidas:,}")
print(f"Formato antes                 : {acidentes_prf_historico.shape}")
print(f"Formato depois                : {df_deduplicado.shape}")

assert removidas == qtd_duplicadas
assert len(df_deduplicado) == len(acidentes_prf_historico) - qtd_duplicadas

if removidas == 0:
    print("\nNenhuma duplicidade exata: a base já estava sem repetição integral de linha.")


Duplicidades exatas encontradas: 0


Linhas removidas              : 0
Formato antes                 : (213497, 31)
Formato depois                : (213497, 31)

Nenhuma duplicidade exata: a base já estava sem repetição integral de linha.


### Duplicidade de chave: `id`

**Pergunta:** o identificador de ocorrência se repete, dentro do ano ou entre anos?

A chave lógica é `(arquivo_origem, id)`, não `id` sozinho. Também registro o caso do `id`
corrompido (`6e+05`, em 2024), que não é duplicata, mas é uma perda de identificador.


In [6]:
chave = acidentes_prf_historico[["arquivo_origem", "id"]]
print(f"Duplicidade em (arquivo_origem, id): {int(chave.duplicated().sum())}")
print(f"Duplicidade em id isolado          : {int(acidentes_prf_historico['id'].duplicated().sum())}")

ids_invalidos = acidentes_prf_historico[
    ~acidentes_prf_historico["id"].fillna("").str.strip().str.fullmatch(r"\d+")
]
print(f"\nIDs não numéricos (sinalizados, NÃO descartados): {len(ids_invalidos)}")
display(ids_invalidos[["id", "data_inversa", "uf", "municipio", "arquivo_origem"]])


Duplicidade em (arquivo_origem, id): 0
Duplicidade em id isolado          : 0

IDs não numéricos (sinalizados, NÃO descartados): 1


,id,data_inversa,uf,municipio,arquivo_origem
104927,6e+05,2024-05-12,PR,SAO JOSE DOS PINHAIS,2024


## 2.2 Conversão de tipos

**Pergunta:** quais colunas são numéricas ou de data e precisam de conversão?

Dois cuidados que o dado exige:

1. **Decimal com vírgula** (`km = "546,2"`, `latitude = "-23,48586772"`).
2. **`data_inversa` é ISO (`yyyy-mm-dd`)**, não `dd/mm/yyyy`. Usar `%d/%m/%Y` transformaria
   as 213.497 datas em `NaT`.

Toda conversão é registrada numa tabela de não-convertidos: se algo falhar, aparece em vez de
virar `NaN` silencioso.


In [7]:
COLUNAS_NUMERICAS = [
    "br", "km", "pessoas", "mortos", "feridos", "feridos_leves",
    "feridos_graves", "ilesos", "ignorados", "veiculos",
    "latitude", "longitude",
]

df_limpo = df_deduplicado.copy()
diagnostico_conversao = []

for coluna in COLUNAS_NUMERICAS:
    original = df_limpo[coluna]
    texto = original.astype("string").str.strip().str.replace(",", ".", regex=False)
    convertido = pd.to_numeric(texto, errors="coerce")
    df_limpo[coluna] = convertido
    diagnostico_conversao.append({
        "coluna": coluna,
        "convertidos": int(convertido.notna().sum()),
        "nao_convertidos": int(convertido.isna().sum() - original.isna().sum()),
    })

df_limpo["data_inversa"] = pd.to_datetime(
    df_limpo["data_inversa"], format="%Y-%m-%d", errors="coerce"
)

diag = pd.DataFrame(diagnostico_conversao)
diag.loc[len(diag)] = {
    "coluna": "data_inversa",
    "convertidos": int(df_limpo["data_inversa"].notna().sum()),
    "nao_convertidos": int(df_limpo["data_inversa"].isna().sum()),
}

display(diag)
assert diag["nao_convertidos"].sum() == 0, "Há valores que não puderam ser convertidos."
print("OK — todas as colunas numéricas e a data foram convertidas sem perda.")


,coluna,convertidos,nao_convertidos
0,br,213497,0
1,km,213497,0
2,pessoas,213497,0
3,mortos,213497,0
4,feridos,213497,0
5,feridos_leves,213497,0
6,feridos_graves,213497,0
7,ilesos,213497,0
8,ignorados,213497,0
9,veiculos,213497,0


OK — todas as colunas numéricas e a data foram convertidas sem perda.


## 2.3 Padronização de textos e validação de domínio

**Pergunta:** existe valor com formatação inválida ou fora do dicionário da PRF?

Esta é a etapa pedida no exercício ("padronize textos, nulos de categorias como `IGNORADO`"),
implementada como **validador**: em vez de só aplicar `strip`/`upper` no escuro, o notebook
confere cada coluna categórica contra o domínio esperado e registra toda substituição.

### 2.3.1 Limpeza de formatação

Remove espaços nas pontas, espaços duplicados no meio e caracteres invisíveis
(espaço não separável, zero-width). Esses defeitos existem na fonte: `municipio` tem
32 linhas com espaço duplo (ex.: `SAO SEBASTIAO DE LAGOA DE  ROCA`).


In [8]:
def limpar_espacos(serie: pd.Series) -> pd.Series:
    """Remove espaço nas pontas, espaço duplicado no meio e caracteres invisíveis."""
    return (
        serie.astype("string")
        .str.replace(r"[\u00a0\u200b\u200e\u200f]", " ", regex=True)
        .str.strip()
        .str.replace(r"\s{2,}", " ", regex=True)
    )


# Em pandas 3, colunas de texto têm dtype 'str' (antes eram 'object');
# 'string' é o StringDtype opcional. As três formas precisam ser aceitas.
def eh_texto(serie: pd.Series) -> bool:
    return str(serie.dtype) in ("object", "string", "str")


COLUNAS_TEXTO = [c for c in df_limpo.columns if eh_texto(df_limpo[c])]
print(f"Colunas de texto detectadas ({len(COLUNAS_TEXTO)}): {COLUNAS_TEXTO}")

registro_formatacao = []
for coluna in COLUNAS_TEXTO:
    antes = df_limpo[coluna]
    depois = limpar_espacos(antes)
    alteradas = int((antes.notna() & (antes != depois)).sum())
    if alteradas:
        registro_formatacao.append({"coluna": coluna, "linhas_alteradas": alteradas})
    df_limpo[coluna] = depois

if registro_formatacao:
    display(pd.DataFrame(registro_formatacao))
else:
    print("Nenhuma alteração de espaçamento necessária.")

print("\nExemplo de município corrigido (se havia espaço duplo):")
exemplo = df_limpo[df_limpo["municipio"].str.contains("LAGOA DE ROCA", na=False)]
display(exemplo[["uf", "municipio"]].drop_duplicates().head(3))
if exemplo.empty:
    print("Nenhum município com esse padrão na base.")


Colunas de texto detectadas (17): ['id', 'dia_semana', 'horario', 'uf', 'municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'regional', 'delegacia', 'uop']


,coluna,linhas_alteradas
0,municipio,32



Exemplo de município corrigido (se havia espaço duplo):


,uf,municipio
18835,PB,SAO SEBASTIAO DE LAGOA DE ROCA


### 2.3.2 Dicionário de domínio das categóricas controladas

Domínio esperado por coluna, conforme o dicionário de dados da PRF. `None` significa
"vocabulário aberto" — nesse caso só se verifica formatação, não pertencimento.

A comparação é tolerante a caixa: um valor como `plena noite` seria **normalizado** para
`Plena Noite` em vez de virar `IGNORADO` — é correção de forma, não de conteúdo.


In [9]:
DOMINIOS = {
    "fase_dia": {"Amanhecer", "Anoitecer", "Plena Noite", "Pleno dia"},
    "tipo_pista": {"Dupla", "Múltipla", "Simples"},
    "uso_solo": {"Não", "Sim"},
    "sentido_via": {"Crescente", "Decrescente", "Não Informado"},
    "classificacao_acidente": {"Com Vítimas Fatais", "Com Vítimas Feridas", "Sem Vítimas"},
    "dia_semana": {
        "domingo", "segunda-feira", "terça-feira", "quarta-feira",
        "quinta-feira", "sexta-feira", "sábado",
    },
    # vocabulário aberto: só formatação
    "causa_acidente": None,
    "tipo_acidente": None,
    "condicao_metereologica": None,
    "tracado_via": None,
    "municipio": None,
}

UF_VALIDAS = {
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS",
    "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC",
    "SE", "SP", "TO",
}
DOMINIOS["uf"] = UF_VALIDAS

placeholder = pd.DataFrame({
    "coluna": list(DOMINIOS),
    "dominio_fechado": [d is not None for d in DOMINIOS.values()],
    "qtd_valores_aceitos": [len(d) if d else np.nan for d in DOMINIOS.values()],
})
display(placeholder)


,coluna,dominio_fechado,qtd_valores_aceitos
0,fase_dia,True,4.0
1,tipo_pista,True,3.0
2,uso_solo,True,2.0
3,sentido_via,True,3.0
4,classificacao_acidente,True,3.0
5,dia_semana,True,7.0
6,causa_acidente,False,NaN
7,tipo_acidente,False,NaN
8,condicao_metereologica,False,NaN
9,tracado_via,False,NaN


### 2.3.3 Validador: `validar_e_corrigir_dominio`

Para cada coluna com domínio fechado:

| Situação | Ação |
|---|---|
| Valor nulo (`NaN`/vazio) | → `IGNORADO` |
| Difere do domínio só por caixa ou acento | → normaliza para o rótulo canônico |
| Valor que não existe no domínio | → `IGNORADO` **e** registra o valor original |
| Valor válido | mantém |

O retorno é um log auditável: quantas linhas mudaram, por qual motivo e qual valor caiu em
`IGNORADO`. Nada é substituído em silêncio.


In [10]:
LITERAL_IGNORADO = "IGNORADO"


def _assinatura(valor: str) -> str:
    """Chave insensível a caixa, acento e pontuação."""
    valor = unicodedata.normalize("NFKD", str(valor))
    valor = valor.encode("ascii", "ignore").decode("utf-8")
    return "".join(ch for ch in valor.lower() if ch.isalnum())


def validar_e_corrigir_dominio(
    base: pd.DataFrame, coluna: str, dominio: set[str] | None, *,
    literal_ignorado: str = LITERAL_IGNORADO,
) -> pd.DataFrame:
    """Valida uma categórica contra o domínio e devolve o log das correções.

    Só atua em coluna com domínio fechado; para vocabulário aberto devolve
    um log vazio (a formatação já foi tratada em 2.3.1).
    """
    if dominio is None:
        return pd.DataFrame(columns=["coluna", "motivo", "linhas", "valores"])

    canonical = {_assinatura(v): v for v in dominio}
    serie = base[coluna]
    novo = serie.copy()
    eventos: dict[str, dict] = {}

    for idx, valor in serie.items():
        if pd.isna(valor) or str(valor).strip() == "":
            novo.at[idx] = literal_ignorado
            eventos.setdefault("nulo → IGNORADO", {}).setdefault("valores", set()).add("<vazio>")
            eventos["nulo → IGNORADO"]["linhas"] = eventos["nulo → IGNORADO"].get("linhas", 0) + 1
            continue
        if valor in dominio:
            continue
        chave = _assinatura(valor)
        if chave in canonical:
            novo.at[idx] = canonical[chave]
            eventos.setdefault("forma normalizada", {}).setdefault("valores", set()).add(
                f"{valor} → {canonical[chave]}"
            )
            eventos["forma normalizada"]["linhas"] = eventos["forma normalizada"].get("linhas", 0) + 1
        else:
            novo.at[idx] = literal_ignorado
            eventos.setdefault("fora do domínio → IGNORADO", {}).setdefault("valores", set()).add(str(valor))
            eventos["fora do domínio → IGNORADO"]["linhas"] = eventos["fora do domínio → IGNORADO"].get("linhas", 0) + 1

    base[coluna] = novo.astype("string")

    linhas_log = [
        {"coluna": coluna, "motivo": motivo,
         "linhas": dados["linhas"],
         "valores": "; ".join(sorted(dados["valores"]))[:120]}
        for motivo, dados in eventos.items()
    ]
    return pd.DataFrame(linhas_log)


### 2.3.4 Teste do validador com valores inválidos injetados

**Pergunta:** o validador realmente detecta problemas, ou apenas passa por tudo?

Um validador que nunca acusa nada não é verificação. Este teste injeta numa cópia os quatro
tipos de defeito (`nulo`, caixa errada, acento faltando, valor inexistente) e confere que cada
um é classificado corretamente — sem tocar na base real.


In [11]:
amostra = pd.DataFrame({
    "fase_dia": [
        "Plena Noite",              # válido
        "plena noite",              # caixa errada -> normaliza
        "Pleno Dia",                # caixa errada -> normaliza para 'Pleno dia'
        None,                       # nulo -> IGNORADO
        "Madrugada",                # inexistente -> IGNORADO
        "Anoitecer ",               # espaço sobrando (já limpo em 2.3.1) -> válido
    ]
})

log_teste = validar_e_corrigir_dominio(amostra, "fase_dia", DOMINIOS["fase_dia"])
display(log_teste)
display(amostra.rename(columns={"fase_dia": "resultado"}))

obtido = amostra["fase_dia"].tolist()
esperado = ["Plena Noite", "Plena Noite", "Pleno dia", "IGNORADO", "IGNORADO", "Anoitecer"]
assert obtido == esperado, f"Validador falhou.\nesperado={esperado}\nobtido={obtido}"
assert len(log_teste) == 3, "Esperava 3 motivos distintos de correção."
print("OK — o validador detecta e classifica os quatro tipos de defeito.")


,coluna,motivo,linhas,valores
0,fase_dia,forma normalizada,3,Anoitecer → Anoitecer; Pleno Dia → Pleno dia;...
1,fase_dia,nulo → IGNORADO,1,<vazio>
2,fase_dia,fora do domínio → IGNORADO,1,Madrugada


,resultado
0,Plena Noite
1,Plena Noite
2,Pleno dia
3,IGNORADO
4,IGNORADO
5,Anoitecer


OK — o validador detecta e classifica os quatro tipos de defeito.


### 2.3.5 Execução do validador em todas as categóricas

**Pergunta:** sobrou algum valor inválido depois da validação?

Agora com a base real. Depois de aplicar, o notebook revalida: nenhuma coluna de domínio
fechado pode conter valor nulo ou fora do dicionário (exceto o próprio `IGNORADO`).


In [12]:
logs = []
for coluna, dominio in DOMINIOS.items():
    if coluna not in df_limpo.columns:
        continue
    log = validar_e_corrigir_dominio(df_limpo, coluna, dominio)
    if not log.empty:
        logs.append(log)

log_dominio = (
    pd.concat(logs, ignore_index=True) if logs
    else pd.DataFrame(columns=["coluna", "motivo", "linhas", "valores"])
)

print(f"Total de substituições registradas: {int(log_dominio['linhas'].sum()) if len(log_dominio) else 0}")
display(log_dominio)


Total de substituições registradas: 3


,coluna,motivo,linhas,valores
0,classificacao_acidente,nulo → IGNORADO,3,<vazio>


In [13]:
falhas = []
for coluna, dominio in DOMINIOS.items():
    if dominio is None or coluna not in df_limpo.columns:
        continue
    presentes = set(df_limpo[coluna].dropna().unique())
    permitidos = set(dominio) | {LITERAL_IGNORADO}
    invalidos = presentes - permitidos
    nulos = int(df_limpo[coluna].isna().sum())
    if invalidos or nulos:
        falhas.append({"coluna": coluna, "fora_do_dominio": sorted(invalidos), "nulos": nulos})
    print(f"{coluna:<24} valores={len(presentes):>2} inválidos={len(invalidos):>2} nulos={nulos}")

assert not falhas, f"Ainda há valores inválidos: {falhas}"
print("\nOK — nenhuma categórica de domínio fechado tem valor inválido ou nulo.")


fase_dia                 valores= 4 inválidos= 0 nulos=0
tipo_pista               valores= 3 inválidos= 0 nulos=0
uso_solo                 valores= 2 inválidos= 0 nulos=0
sentido_via              valores= 3 inválidos= 0 nulos=0
classificacao_acidente   valores= 4 inválidos= 0 nulos=0
dia_semana               valores= 7 inválidos= 0 nulos=0
uf                       valores=27 inválidos= 0 nulos=0

OK — nenhuma categórica de domínio fechado tem valor inválido ou nulo.


### 2.3.6 Nulos das contagens → 0

**Pergunta:** as contagens de vítimas têm nulos?

Nenhuma tem (a base é completa), mas a regra é aplicada e verificada: `0` é o valor correto
para "nenhuma vítima", e um `NaN` inflaria as médias e quebraria somas.


In [14]:
COLUNAS_CONTAGEM = ["pessoas", "mortos", "feridos", "feridos_leves",
                    "feridos_graves", "ilesos", "ignorados", "veiculos"]

nulos_antes = {c: int(df_limpo[c].isna().sum()) for c in COLUNAS_CONTAGEM}
for c in COLUNAS_CONTAGEM:
    df_limpo[c] = df_limpo[c].fillna(0).astype("int64")
nulos_depois = {c: int(df_limpo[c].isna().sum()) for c in COLUNAS_CONTAGEM}

display(pd.DataFrame({"nulos_antes": nulos_antes, "nulos_depois": nulos_depois}))
assert sum(nulos_depois.values()) == 0
print("OK — contagens sem nulos e tipadas como inteiro.")


,nulos_antes,nulos_depois
pessoas,0,0
mortos,0,0
feridos,0,0
feridos_leves,0,0
feridos_graves,0,0
ilesos,0,0
ignorados,0,0
veiculos,0,0


OK — contagens sem nulos e tipadas como inteiro.


## 2.4 Remoção das colunas de localidade fina

**Pergunta:** quais colunas saem e o que permanece para localizar o acidente?

Saem `latitude`, `longitude`, `regional`, `delegacia` e `uop` (localidade administrativa fina,
com muitos nulos e sem uso nas análises). Permanecem `uf`, `br` e `municipio`, que são os
níveis de agregação usados nas questões de negócio.


In [15]:
COLUNAS_REMOVER = ["latitude", "longitude", "regional", "delegacia", "uop"]
presentes = [c for c in COLUNAS_REMOVER if c in df_limpo.columns]

formato_antes = df_limpo.shape
df_limpo = df_limpo.drop(columns=presentes)
formato_depois = df_limpo.shape

print(f"Removidas : {presentes}")
print(f"Formato   : {formato_antes} → {formato_depois}")

for obrigatoria in ("uf", "br", "municipio"):
    assert obrigatoria in df_limpo.columns, f"'{obrigatoria}' deveria ter sido mantida."
for removida in COLUNAS_REMOVER:
    assert removida not in df_limpo.columns, f"'{removida}' deveria ter sido removida."

print(f"\nColunas finais ({len(df_limpo.columns)}): {sorted(df_limpo.columns)}")


Removidas : ['latitude', 'longitude', 'regional', 'delegacia', 'uop']
Formato   : (213497, 31) → (213497, 26)

Colunas finais (26): ['arquivo_origem', 'br', 'causa_acidente', 'classificacao_acidente', 'condicao_metereologica', 'data_inversa', 'dia_semana', 'fase_dia', 'feridos', 'feridos_graves', 'feridos_leves', 'horario', 'id', 'ignorados', 'ilesos', 'km', 'mortos', 'municipio', 'pessoas', 'sentido_via', 'tipo_acidente', 'tipo_pista', 'tracado_via', 'uf', 'uso_solo', 'veiculos']


## 2.5 Fechamento da Parte 2

**Pergunta:** a base limpa preservou todas as ocorrências?

A remoção de colunas muda a largura, nunca a altura. O `assert` garante que nenhuma linha foi
perdida na limpeza.


In [16]:
assert len(df_limpo) == linhas_na_integracao, "A limpeza alterou o número de linhas."

resumo = pd.DataFrame([
    {"etapa": "base consolidada (Parte 1)", "linhas": linhas_na_integracao,
     "colunas": acidentes_prf_historico.shape[1]},
    {"etapa": "após deduplicação (2.1)", "linhas": len(df_deduplicado),
     "colunas": df_deduplicado.shape[1]},
    {"etapa": "df_limpo final (2.5)", "linhas": len(df_limpo),
     "colunas": df_limpo.shape[1]},
])
display(resumo)

print(f"Acidentes fatais preservados: {int((df_limpo['mortos'] >= 1).sum()):,}")
print(f"Total de mortos preservado  : {int(df_limpo['mortos'].sum()):,}")


,etapa,linhas,colunas
0,base consolidada (Parte 1),213497,31
1,após deduplicação (2.1),213497,31
2,df_limpo final (2.5),213497,26


Acidentes fatais preservados: 15,293
Total de mortos preservado  : 17,833
